[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/01_ONNX_with_Python/01_Linear_Regression_Example/Linear_Regression_Example_Deep_Dive.ipynb)

# 1.1 Linear Regression in ONNX — Deep Dive

Build your **first ONNX computation graph** from scratch using the Python helper API.

---

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [Mathematical Formulation](#section-1) | The linear regression equation and its decomposition |
| 2 | [ONNX Graph Architecture](#section-2) | How ONNX represents computations as directed acyclic graphs |
| 3 | [The Four Core Helper Functions](#section-3) | `make_tensor_value_info`, `make_node`, `make_graph`, `make_model` |
| 4 | [Step-by-Step Graph Construction](#section-4) | Building the linear regression graph with code |
| 5 | [Visualizing the Computation Graph](#section-5) | Matplotlib-based graph visualization |
| 6 | [Inspecting the Protobuf Representation](#section-6) | Reading nodes, inputs, outputs, and edges programmatically |
| 7 | [Worked Example: Numerical Walkthrough](#section-7) | Tracing actual values through the graph |
| 8 | [Dynamic vs Static Shapes](#section-8) | Understanding `None` dimensions and shape semantics |
| 9 | [Key Takeaways](#section-9) | Summary and interview reference |

<a id='section-1'></a>
## Section 1: Mathematical Formulation

### The Linear Regression Equation

Linear regression models the relationship between an input matrix $X$ and an output vector $Y$ through a weight matrix $A$ and a bias vector $B$:

$$Y = XA + B$$

where:

| Symbol | Shape | Description |
|--------|-------|-------------|
| $X$ | $(N, D)$ | Input feature matrix — $N$ samples, $D$ features |
| $A$ | $(D, K)$ | Weight (coefficient) matrix |
| $B$ | $(1, K)$ or $(K,)$ | Bias (intercept) vector, broadcast across rows |
| $Y$ | $(N, K)$ | Predicted output — $N$ samples, $K$ targets |

### Decomposition into Primitive Operations

ONNX does not have a single "LinearRegression" operator in its standard set. Instead, we decompose the formula into two **primitive operations** that ONNX *does* support:

**Step 1 — Matrix Multiplication:**

$$T = \text{MatMul}(X, A) \quad \Longrightarrow \quad T_{ij} = \sum_{d=1}^{D} X_{id} \cdot A_{dj}$$

**Step 2 — Element-wise Addition (with broadcast):**

$$Y = \text{Add}(T, B) \quad \Longrightarrow \quad Y_{ij} = T_{ij} + B_j$$

The bias $B$ is broadcast along the batch dimension (axis 0), so a single bias vector is added to every row of $T$. This broadcasting follows the standard NumPy rules that ONNX adopts.

### Why This Decomposition Matters

Every ONNX model — whether it represents a simple linear regression or a billion-parameter transformer — is built from exactly this kind of decomposition. Complex models are just larger graphs of these same primitive operators. Understanding how to wire `MatMul` and `Add` into a graph gives you the foundational skill for building *any* ONNX model.

The key insight is the **Single Static Assignment (SSA)** naming convention: each intermediate result gets a unique name (like `XA`), and downstream nodes reference that name as an input. This creates the directed edges of the computation graph.

<a id='section-2'></a>
## Section 2: ONNX Graph Architecture

An ONNX model is fundamentally a **directed acyclic graph (DAG)**. Every computation flows forward from inputs to outputs, with no cycles. The graph has three types of components:

```
┌─────────────────────────────────────────────────────────┐
│                    ModelProto                           │
│  ┌───────────────────────────────────────────────────┐  │
│  │                  GraphProto                       │  │
│  │                                                   │  │
│  │   INPUTS (ValueInfoProto)                         │  │
│  │   ┌─────┐   ┌─────┐   ┌─────┐                    │  │
│  │   │  X  │   │  A  │   │  B  │                    │  │
│  │   │NxD  │   │DxK  │   │1xK  │                    │  │
│  │   └──┬──┘   └──┬──┘   └──┬──┘                    │  │
│  │      │         │         │                        │  │
│  │      ▼         ▼         │                        │  │
│  │   ┌────────────────┐     │                        │  │
│  │   │   MatMul       │     │    NODES (NodeProto)   │  │
│  │   │  inputs: X, A  │     │                        │  │
│  │   │  output: XA    │     │                        │  │
│  │   └───────┬────────┘     │                        │  │
│  │           │              │                        │  │
│  │           ▼              ▼                        │  │
│  │   ┌────────────────────────┐                      │  │
│  │   │        Add             │                      │  │
│  │   │  inputs: XA, B        │                      │  │
│  │   │  output: Y            │                      │  │
│  │   └───────────┬────────────┘                      │  │
│  │               │                                   │  │
│  │               ▼                                   │  │
│  │   ┌─────────────┐                                 │  │
│  │   │      Y      │  OUTPUT (ValueInfoProto)        │  │
│  │   │    NxK      │                                 │  │
│  │   └─────────────┘                                 │  │
│  └───────────────────────────────────────────────────┘  │
│  opset_import: [ai.onnx v15]                           │
│  ir_version: 8                                          │
└─────────────────────────────────────────────────────────┘
```

### The Three Component Types

| Component | Protobuf Type | Role | Analogy |
|-----------|--------------|------|----------|
| **Inputs / Outputs** | `ValueInfoProto` | Declare typed tensor interfaces | Function signature parameters |
| **Nodes** | `NodeProto` | Execute operations on named tensors | Function body statements |
| **Edges** | *(implicit)* | Connect nodes via matching names | Variable references |

### Edge Wiring via Name Matching

ONNX graphs do **not** have explicit edge objects. Instead, edges are *implied* by name matching: if node A produces an output called `"XA"` and node B lists `"XA"` as one of its inputs, there is an edge from A to B carrying that tensor. This is the SSA (Single Static Assignment) pattern — each name is produced by exactly one node and can be consumed by one or more downstream nodes.

This design has important consequences:
- **Names must be unique** within a scope (no two nodes can produce the same output name).
- **Topological order** of nodes must respect data dependencies (a node cannot consume a name before it is produced).
- **Dead code** is possible: if a node's output name is never consumed by another node or listed as a graph output, that node is effectively dead.

In [ ]:
# Install dependencies (uncomment if running in Colab)
# !pip install onnx onnxruntime matplotlib numpy

<a id='section-3'></a>
## Section 3: The Four Core Helper Functions

ONNX provides a `helper` module with factory functions that construct protobuf objects. For building any graph, you need exactly four functions:

### 3.1 `make_tensor_value_info(name, elem_type, shape)`

Creates a `ValueInfoProto` — a typed tensor declaration used for graph inputs and outputs.

| Parameter | Type | Description |
|-----------|------|-------------|
| `name` | `str` | Unique identifier for this tensor |
| `elem_type` | `int` | Data type from `TensorProto` enum (e.g., `FLOAT`, `INT64`) |
| `shape` | `list` | Dimension sizes; `None` means dynamic |

A `ValueInfoProto` does **not** contain data — it only declares the *type signature*. Think of it as a function parameter declaration: it says "this slot expects a float32 tensor of shape $(N, D)$" without providing actual values.

### 3.2 `make_node(op_type, inputs, outputs, **attributes)`

Creates a `NodeProto` — a single operation in the graph.

| Parameter | Type | Description |
|-----------|------|-------------|
| `op_type` | `str` | Operator name from the ONNX spec (e.g., `"MatMul"`, `"Add"`) |
| `inputs` | `list[str]` | Names of input tensors (must match upstream outputs) |
| `outputs` | `list[str]` | Names of output tensors (must be unique in the graph) |
| `**attributes` | keyword args | Fixed operator parameters (e.g., `perm=[1,0]` for Transpose) |

### 3.3 `make_graph(nodes, name, inputs, outputs, initializers=[])`

Assembles nodes and interfaces into a `GraphProto`.

| Parameter | Type | Description |
|-----------|------|-------------|
| `nodes` | `list[NodeProto]` | Operations in topological order |
| `name` | `str` | Graph name |
| `inputs` | `list[ValueInfoProto]` | Dynamic inputs (provided at runtime) |
| `outputs` | `list[ValueInfoProto]` | What the graph produces |
| `initializers` | `list[TensorProto]` | Constant weights baked into the model |

### 3.4 `make_model(graph, **kwargs)`

Wraps a graph into a `ModelProto` — the top-level container that includes metadata, opset declarations, and the graph itself.

```
make_tensor_value_info ──► ValueInfoProto ─┐
                                           │
make_node ──────────────► NodeProto ───────┼──► make_graph ──► GraphProto ──► make_model ──► ModelProto
                                           │
make_tensor_value_info ──► ValueInfoProto ─┘
```

<a id='section-4'></a>
## Section 4: Step-by-Step Graph Construction

We now build the linear regression graph $Y = XA + B$ using the four helper functions. Each step corresponds to one layer of the architecture diagram above.

In [ ]:
from onnx import TensorProto
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info)
from onnx.checker import check_model

# ── Step 1: Declare inputs ──────────────────────────────────────────
# Each input is a ValueInfoProto with (name, dtype, shape).
# Using [None, None] means both dimensions are dynamic.

X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])  # (N, D)
A = make_tensor_value_info('A', TensorProto.FLOAT, [None, None])  # (D, K)
B = make_tensor_value_info('B', TensorProto.FLOAT, [None, None])  # (1, K) or (K,)

# ── Step 2: Declare output ──────────────────────────────────────────
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])        # (N, K)

print(f'Input X: name={X.name!r}, type={X.type}')
print(f'Output Y: name={Y.name!r}, type={Y.type}')

In [ ]:
# ── Step 3: Create operator nodes ───────────────────────────────────
#
# Node 1: T = MatMul(X, A)   — matrix multiplication
# Node 2: Y = Add(T, B)     — element-wise addition with broadcast
#
# The string 'XA' is the SSA name that wires node1's output to node2's input.

node_matmul = make_node('MatMul', ['X', 'A'], ['XA'])
node_add    = make_node('Add',    ['XA', 'B'], ['Y'])

print(f'Node 1: op={node_matmul.op_type}, inputs={list(node_matmul.input)}, outputs={list(node_matmul.output)}')
print(f'Node 2: op={node_add.op_type}, inputs={list(node_add.input)}, outputs={list(node_add.output)}')

In [ ]:
# ── Step 4: Assemble graph and model ────────────────────────────────

graph = make_graph(
    [node_matmul, node_add],   # nodes in topological order
    'linear_regression',       # graph name
    [X, A, B],                 # inputs
    [Y]                        # outputs
)

onnx_model = make_model(graph)
check_model(onnx_model)  # validates structural correctness

print('Model built and validated successfully!')
print(f'IR version: {onnx_model.ir_version}')
print(f'OpSet: {onnx_model.opset_import[0].domain or "ai.onnx"} v{onnx_model.opset_import[0].version}')
print(f'Number of nodes: {len(onnx_model.graph.node)}')
print(f'Number of inputs: {len(onnx_model.graph.input)}')
print(f'Number of outputs: {len(onnx_model.graph.output)}')

<a id='section-5'></a>
## Section 5: Visualizing the Computation Graph

A visual representation makes it much easier to understand how data flows through the graph. Below we use **matplotlib** to render the computation graph, showing inputs as rounded rectangles, operators as boxes, and edges as arrows.

This is the kind of visualization that tools like **Netron** produce for ONNX models, but here we build it programmatically so you can see exactly how the graph structure maps to a picture.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(1, 1, figsize=(8, 7))
ax.set_xlim(-1, 7)
ax.set_ylim(-1, 7)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('ONNX Computation Graph: Y = XA + B', fontsize=14, fontweight='bold')

input_style = dict(boxstyle='round,pad=0.4', facecolor='#AED6F1', edgecolor='#2C3E50', linewidth=2)
op_style = dict(boxstyle='square,pad=0.4', facecolor='#F9E79F', edgecolor='#7D6608', linewidth=2)
output_style = dict(boxstyle='round,pad=0.4', facecolor='#A9DFBF', edgecolor='#1E8449', linewidth=2)
inter_style = dict(boxstyle='round,pad=0.3', facecolor='#FADBD8', edgecolor='#C0392B', linewidth=1.5)

# Inputs
ax.text(1, 6, 'X\n(N×D)', ha='center', va='center', fontsize=11, fontweight='bold', bbox=input_style)
ax.text(3, 6, 'A\n(D×K)', ha='center', va='center', fontsize=11, fontweight='bold', bbox=input_style)
ax.text(5, 4, 'B\n(1×K)', ha='center', va='center', fontsize=11, fontweight='bold', bbox=input_style)

# Operators
ax.text(2, 4, 'MatMul', ha='center', va='center', fontsize=12, fontweight='bold', bbox=op_style)
ax.text(3, 2, 'Add', ha='center', va='center', fontsize=12, fontweight='bold', bbox=op_style)

# Intermediate
ax.text(2, 3, 'XA\n(N×K)', ha='center', va='center', fontsize=9, bbox=inter_style)

# Output
ax.text(3, 0.5, 'Y\n(N×K)', ha='center', va='center', fontsize=11, fontweight='bold', bbox=output_style)

# Arrows
arrow_kw = dict(arrowstyle='->', color='#2C3E50', lw=2, mutation_scale=15)
ax.annotate('', xy=(2, 4.6), xytext=(1, 5.4), arrowprops=arrow_kw)
ax.annotate('', xy=(2, 4.6), xytext=(3, 5.4), arrowprops=arrow_kw)
ax.annotate('', xy=(2, 3.4), xytext=(2, 3.6), arrowprops=arrow_kw)
ax.annotate('', xy=(3, 2.6), xytext=(2, 2.6), arrowprops=arrow_kw)
ax.annotate('', xy=(3, 2.6), xytext=(5, 3.4), arrowprops=arrow_kw)
ax.annotate('', xy=(3, 1.1), xytext=(3, 1.5), arrowprops=arrow_kw)

# Legend
legend_elements = [
    mpatches.Patch(facecolor='#AED6F1', edgecolor='#2C3E50', label='Input (ValueInfoProto)'),
    mpatches.Patch(facecolor='#F9E79F', edgecolor='#7D6608', label='Operator (NodeProto)'),
    mpatches.Patch(facecolor='#FADBD8', edgecolor='#C0392B', label='Intermediate Tensor'),
    mpatches.Patch(facecolor='#A9DFBF', edgecolor='#1E8449', label='Output (ValueInfoProto)')]
ax.legend(handles=legend_elements, loc='lower left', fontsize=9)

plt.tight_layout()
plt.show()

<a id='section-6'></a>
## Section 6: Inspecting the Protobuf Representation

Every ONNX object is a Protocol Buffer (protobuf) message. This means you can programmatically inspect any part of the model. Understanding the protobuf structure is essential for debugging, as it reveals exactly how your model is stored on disk.

### The Protobuf Hierarchy

```
ModelProto
├── ir_version: int
├── opset_import: [OpSetIdProto, ...]
│   ├── domain: str
│   └── version: int
├── producer_name: str
└── graph: GraphProto
    ├── name: str
    ├── input: [ValueInfoProto, ...]
    │   ├── name: str
    │   └── type: TypeProto
    │       └── tensor_type: TensorTypeProto
    │           ├── elem_type: int (1=FLOAT, 7=INT64, ...)
    │           └── shape: TensorShapeProto
    │               └── dim: [Dimension, ...]
    │                   ├── dim_value: int (static)
    │                   └── dim_param: str (symbolic)
    ├── output: [ValueInfoProto, ...]
    ├── node: [NodeProto, ...]
    │   ├── op_type: str
    │   ├── input: [str, ...]
    │   ├── output: [str, ...]
    │   └── attribute: [AttributeProto, ...]
    └── initializer: [TensorProto, ...]
```

Let's write utility functions to walk this tree and extract useful information.

In [ ]:
def shape_to_tuple(shape):
    """Convert a TensorShapeProto to a readable tuple."""
    dims = []
    for d in shape.dim:
        if d.dim_param:        # symbolic (e.g., 'N', 'batch')
            dims.append(d.dim_param)
        elif d.dim_value > 0:  # static
            dims.append(d.dim_value)
        else:                  # dynamic (None)
            dims.append('?')
    return tuple(dims)

DTYPE_MAP = {1: 'float32', 2: 'uint8', 3: 'int8', 5: 'float16',
             6: 'int32', 7: 'int64', 9: 'bool', 10: 'float16',
             11: 'float64', 12: 'uint32', 13: 'uint64'}

print('=' * 60)
print('MODEL INSPECTION REPORT')
print('=' * 60)

print(f'\nIR Version : {onnx_model.ir_version}')
print(f'Graph Name : {onnx_model.graph.name}')
for op in onnx_model.opset_import:
    print(f'OpSet      : domain={op.domain or "ai.onnx"!r}, version={op.version}')

print('\n── INPUTS ─────────────────────────────────────────────────')
for inp in onnx_model.graph.input:
    tt = inp.type.tensor_type
    dtype = DTYPE_MAP.get(tt.elem_type, f'type_{tt.elem_type}')
    shape = shape_to_tuple(tt.shape)
    print(f'  {inp.name:10s}  dtype={dtype:8s}  shape={shape}')

print('\n── NODES ──────────────────────────────────────────────────')
for i, node in enumerate(onnx_model.graph.node):
    print(f'  [{i}] {node.op_type:10s}  inputs={list(node.input)}  →  outputs={list(node.output)}')

print('\n── OUTPUTS ────────────────────────────────────────────────')
for out in onnx_model.graph.output:
    tt = out.type.tensor_type
    dtype = DTYPE_MAP.get(tt.elem_type, f'type_{tt.elem_type}')
    shape = shape_to_tuple(tt.shape)
    print(f'  {out.name:10s}  dtype={dtype:8s}  shape={shape}')

### Understanding the Edge Map

Let's build an explicit edge list to see how the graph is wired. For every tensor name in the model, we record who produces it and who consumes it.

In [ ]:
producers = {}  # tensor_name -> (node_index, op_type) or 'INPUT'
consumers = {}  # tensor_name -> [(node_index, op_type), ...]

for inp in onnx_model.graph.input:
    producers[inp.name] = ('INPUT', inp.name)

for i, node in enumerate(onnx_model.graph.node):
    for out_name in node.output:
        producers[out_name] = (f'node[{i}]', node.op_type)
    for in_name in node.input:
        consumers.setdefault(in_name, []).append((f'node[{i}]', node.op_type))

for out in onnx_model.graph.output:
    consumers.setdefault(out.name, []).append(('OUTPUT', out.name))

print('Edge Map (tensor_name: producer → consumers):')
print('-' * 55)
for name in sorted(set(list(producers.keys()) + list(consumers.keys()))):
    prod = producers.get(name, ('???', '???'))
    cons = consumers.get(name, [])
    cons_str = ', '.join(f'{c[1]}' for c in cons)
    print(f'  {name:5s} :  {prod[1]:8s}  →  {cons_str}')

<a id='section-7'></a>
## Section 7: Worked Example — Numerical Walkthrough

Let's trace actual numerical values through the graph step by step. This is exactly what a runtime does when it executes the model.

### Given

$$X = \begin{bmatrix} 1 & 2 \\ 3 & 4 \\ 5 & 6 \end{bmatrix}_{3 \times 2}, \quad
A = \begin{bmatrix} 0.5 \\ -0.3 \end{bmatrix}_{2 \times 1}, \quad
B = \begin{bmatrix} 1.0 \end{bmatrix}_{1 \times 1}$$

### Find: $Y = XA + B$

**Step 1 — MatMul:** Compute $T = XA$

$$T = \begin{bmatrix} 1 & 2 \\ 3 & 4 \\ 5 & 6 \end{bmatrix} \begin{bmatrix} 0.5 \\ -0.3 \end{bmatrix}
= \begin{bmatrix} 1 \cdot 0.5 + 2 \cdot (-0.3) \\ 3 \cdot 0.5 + 4 \cdot (-0.3) \\ 5 \cdot 0.5 + 6 \cdot (-0.3) \end{bmatrix}
= \begin{bmatrix} -0.1 \\ 0.3 \\ 0.7 \end{bmatrix}$$

**Step 2 — Add (with broadcast):** Compute $Y = T + B$

$$Y = \begin{bmatrix} -0.1 \\ 0.3 \\ 0.7 \end{bmatrix} + \begin{bmatrix} 1.0 \end{bmatrix}
= \begin{bmatrix} 0.9 \\ 1.3 \\ 1.7 \end{bmatrix}$$

Let's verify this with both ONNX Runtime and NumPy:

In [ ]:
import numpy as np
import onnxruntime as ort

x = np.array([[1, 2], [3, 4], [5, 6]], dtype=np.float32)
a = np.array([[0.5], [-0.3]], dtype=np.float32)
b = np.array([[1.0]], dtype=np.float32)

# Run with ONNX Runtime
sess = ort.InferenceSession(
    onnx_model.SerializeToString(),
    providers=['CPUExecutionProvider'])

onnx_result = sess.run(None, {'X': x, 'A': a, 'B': b})[0]

# Verify with NumPy
step1_matmul = x @ a
step2_add = step1_matmul + b

print('Step 1 — MatMul(X, A):')
print(f'  T = {step1_matmul.flatten()}')
print()
print('Step 2 — Add(T, B):')
print(f'  Y = {step2_add.flatten()}')
print()
print(f'ONNX Runtime result: {onnx_result.flatten()}')
print(f'NumPy verification:  {step2_add.flatten()}')
print(f'Match: {np.allclose(onnx_result, step2_add)}')

### Visualizing the Data Flow with Actual Values

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

matrices = [
    ('X (3×2)', x),
    ('A (2×1)', a),
    ('T = XA (3×1)', step1_matmul),
    ('Y = T+B (3×1)', step2_add)
]

cmaps = ['Blues', 'Oranges', 'Purples', 'Greens']

for ax, (title, mat), cmap in zip(axes, matrices, cmaps):
    im = ax.imshow(mat, cmap=cmap, aspect='auto')
    ax.set_title(title, fontsize=11, fontweight='bold')
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f'{mat[i, j]:.1f}', ha='center', va='center',
                    fontsize=12, fontweight='bold')
    ax.set_xticks(range(mat.shape[1]))
    ax.set_yticks(range(mat.shape[0]))
    ax.set_xticklabels([f'col{j}' for j in range(mat.shape[1])])
    ax.set_yticklabels([f'row{i}' for i in range(mat.shape[0])])

# Draw arrows between subplots
fig.text(0.30, 0.5, '×', fontsize=24, fontweight='bold', ha='center', va='center')
fig.text(0.52, 0.5, '→', fontsize=24, fontweight='bold', ha='center', va='center')
fig.text(0.73, 0.5, '+ B →', fontsize=14, fontweight='bold', ha='center', va='center')

fig.suptitle('Numerical Walkthrough: Y = XA + B', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

<a id='section-8'></a>
## Section 8: Dynamic vs Static Shapes

When we declared `X` with shape `[None, None]`, we told ONNX that **both dimensions are dynamic** — they can be any size at runtime. This is a crucial design decision.

### Shape Specification Options

| Declaration | Meaning | Use Case |
|-------------|---------|----------|
| `[None, None]` | Fully dynamic | Flexible batch + feature count |
| `[None, 4]` | Dynamic batch, 4 features | Known feature dim, variable batch |
| `[32, 4]` | Fixed 32×4 | Static deployment (enables optimizations) |
| `['N', 4]` | Symbolic batch, 4 features | Named dims for documentation |

### Trade-offs

**Dynamic shapes** offer maximum flexibility: you can run the same model with batch size 1 for real-time inference or batch size 1024 for throughput testing. However, fully dynamic shapes prevent certain graph optimizations (like memory pre-allocation or operator fusion based on known dimensions).

**Static shapes** enable more aggressive optimization: the runtime can pre-allocate exact buffer sizes, fuse operators based on known shapes, and even select specialized kernel implementations. The downside is that the model can *only* accept inputs matching those exact dimensions.

**Symbolic dimensions** (like `'N'` or `'batch'`) are a middle ground: they're dynamic but carry semantic information. When two inputs share a symbolic dimension name (e.g., both have dimension `'batch'`), the runtime knows those dimensions will always be equal at runtime, enabling some optimizations without fixing the actual size.

Let's see how the same model behaves with different input sizes:

In [ ]:
test_cases = [
    ('Single sample',    np.random.randn(1, 2).astype(np.float32)),
    ('Small batch (4)',  np.random.randn(4, 2).astype(np.float32)),
    ('Large batch (100)', np.random.randn(100, 2).astype(np.float32)),
]

a_fixed = np.array([[0.5], [-0.3]], dtype=np.float32)
b_fixed = np.array([[1.0]], dtype=np.float32)

print('Dynamic shapes allow variable batch sizes with the SAME model:')
print('-' * 60)

for name, x_test in test_cases:
    result = sess.run(None, {'X': x_test, 'A': a_fixed, 'B': b_fixed})[0]
    expected = x_test @ a_fixed + b_fixed
    print(f'{name:20s}  input_shape={str(x_test.shape):10s}  '
          f'output_shape={str(result.shape):10s}  '
          f'correct={np.allclose(result, expected)}')

<a id='section-9'></a>
## Section 9: Key Takeaways

### The Four-Function Pattern

Every ONNX model — from a two-node linear regression to a billion-parameter LLM — is built using the same four functions:

| Step | Function | Creates | Purpose |
|------|----------|---------|----------|
| 1 | `make_tensor_value_info()` | `ValueInfoProto` | Declare typed tensor interfaces |
| 2 | `make_node()` | `NodeProto` | Define operations with SSA wiring |
| 3 | `make_graph()` | `GraphProto` | Assemble nodes into a DAG |
| 4 | `make_model()` | `ModelProto` | Wrap with metadata + opset version |

### Critical Concepts

1. **SSA Naming**: Each tensor gets a unique name; edges are implied by name matching between node outputs and inputs.

2. **Protobuf Serialization**: The entire model is a protobuf message that can be serialized to bytes and saved to a `.onnx` file.

3. **`check_model()`**: Always call this after construction. It catches wiring errors, type mismatches, and unknown operators before you attempt inference.

4. **Dynamic Shapes**: Using `None` for dimensions allows runtime flexibility but may limit optimizations. Choose the right balance for your deployment scenario.

5. **Operator Decomposition**: Complex computations are expressed as compositions of primitive ONNX operators. Understanding this decomposition is the key to building custom models.

---

**Next:** [Serialization](../02_Serialization/) — Learn how to save, load, and inspect ONNX models on disk.